In [52]:
from BayesianFNN import BayesianFNN
import random
import numpy as np
import torch
import os
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor
from tqdm import tqdm
import torch.optim as optim
import torch.nn as nn
import copy
import pandas as pd
import importlib
import matplotlib.pyplot as plt

In [53]:
import BayesianFNN

importlib.reload(BayesianFNN)
from BayesianFNN import BayesianFNN  # re-import

In [3]:
# Set all random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

In [4]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Random seed set to: {SEED} for full reproducibility")

Using device: cuda
Random seed set to: 42 for full reproducibility


In [5]:
def seed_worker(worker_id):
    """Function to ensure DataLoader workers use different seeds derived from the base seed"""
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),  
    transforms.Lambda(lambda x: x.view(-1)) 
])

training_data = datasets.FashionMNIST(
    root="../../Datasets",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.FashionMNIST(
    root="../../Datasets",
    train=False,
    download=True,
    transform=transform
)

In [12]:
def plot_metrics(metrics_dict, save_path='./results/metrics_comparison.png'):
    """Plot comparison of metrics across all models"""
    # Define colors for each model
    colors = {
        'baseline': 'blue',
        'plasticity': 'red',
        # 'dropin_unfrozen': 'green'
    }
    
    # Create figure with subplots
    fig, axs = plt.subplots(3, 3, figsize=(15, 12))
    
    # Training loss total
    ax = axs[0, 0]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['train_loss_total']))
        ax.plot(epochs, metrics['train_loss_total'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Training Loss (nll + kl)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()

    # Training loss nll
    ax = axs[0, 1]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['train_loss_nll']))
        ax.plot(epochs, metrics['train_loss_nll'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Training Loss (nll)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()

    # Training loss nll
    ax = axs[0, 2]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['train_loss_kl']))
        ax.plot(epochs, metrics['train_loss_kl'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Training Loss (kl)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()
    
    # Validation loss total
    ax = axs[1, 0]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['val_loss_total']))
        ax.plot(epochs, metrics['val_loss_total'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Validation Loss (nll + kl)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()

    # Validation loss nll
    ax = axs[1, 1]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['val_loss_nll']))
        ax.plot(epochs, metrics['val_loss_nll'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Validation Loss (nll)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()

    # Validation loss kl
    ax = axs[1, 2]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['val_loss_kl']))
        ax.plot(epochs, metrics['val_loss_kl'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Validation Loss (kl)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()
    
    # Training accuracy
    ax = axs[2, 0]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['train_acc']))
        ax.plot(epochs, metrics['train_acc'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Training Accuracy')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Accuracy (%)')
    ax.legend()
    
    # Validation accuracy
    ax = axs[2, 1]
    for model_name, metrics in metrics_dict.items():
        epochs = range(1, 1 + len(metrics['val_acc']))
        ax.plot(epochs, metrics['val_acc'], color=colors.get(model_name, 'gray'), label=model_name)
    ax.set_title('Validation Accuracy')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Accuracy (%)')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

In [13]:
class EarlyStopping:
    def __init__(self, patience=3, delta=0.025, verbose=True):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.best_loss = None
        self.no_improvement_count = 0
        self.stop_training = False
    
    def check_early_stop(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1
            if self.no_improvement_count >= self.patience:
                self.stop_training = True
                if self.verbose:
                    print("Stopping early as no improvement has been observed.")

In [14]:
def loss_function(outputs, labels, kl_loss, beta=0.5):
    criterion = nn.CrossEntropyLoss()
    nll = criterion(outputs, labels)
    # normalise to per sample
    return nll + kl_loss*beta, nll, kl_loss*beta

In [87]:
def train(model, train_dataloader, optimizer, epoch, device, warmup_epochs=50):
    model.train()
    running_loss_total = 0.0
    running_loss_nll= 0.0
    running_loss_kl = 0.0
    correct = 0
    total = 0
    beta = 1/len(train_dataloader.dataset)
    
    progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch}')
    
    for inputs, labels in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss, nll, kl = loss_function(outputs, labels, model.kl_loss(), beta = beta)
        loss.backward()
        optimizer.step()
        
        # Track statistics
        running_loss_total += loss.item()
        running_loss_nll += nll.item()
        running_loss_kl += kl.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': running_loss_total / (progress_bar.n + 1),
            'acc': 100. * correct / total,
        })
    train_loss_total = running_loss_total / len(train_dataloader)
    train_acc = 100. * correct / total
    train_loss_nll = running_loss_nll / len(train_dataloader)
    train_loss_kl = running_loss_kl / len(train_dataloader)
    
    return train_loss_total, train_acc, train_loss_nll, train_loss_kl

In [121]:
def validate(model, val_dataloader, device):
    model.eval()
    val_loss_total = 0.0
    val_loss_nll = 0.0
    val_loss_kl = 0.0
    correct = 0
    total = 0
    beta = 1/len(val_dataloader.dataset)

    with torch.no_grad():
        for inputs, labels in tqdm(val_dataloader, desc='Validating'):
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss, nll, kl = loss_function(outputs, labels, model.kl_loss(), beta = beta)
            val_loss_total += loss.item()
            val_loss_nll += loss.item()
            val_loss_kl += loss.item()
            
            # Get predicted classes
            _, predicted = outputs.max(1)
            
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss_total = val_loss_total / len(val_dataloader)
    val_loss_nll = val_loss_nll / len(val_dataloader)
    val_loss_kl = val_loss_kl / len(val_dataloader)
    val_acc = 100. * correct / total
    
    return val_loss_total, val_acc, val_loss_nll, val_loss_kl


In [149]:
def snr_based_neurogenesis(plasticity_original, hidden_sizes, neurons_to_add=16):
    snr = plasticity_original.get_average_snr_per_layer()
    print("\n Average Signal-to-Noise Ratio per Hidden Layer:")
    for i, val in enumerate(snr):
        print(f"  Layer {i+1}: {val.item():.4f}")
    layer_to_expand = snr.index(min(snr[1:]))
    print(f"Expanding Layer {layer_to_expand+1} "
          f"(lowest SNR: {snr[layer_to_expand].item():.4f}) "
          f"by {neurons_to_add} neurons")
    expanded_hidden_sizes = hidden_sizes.copy()
    expanded_hidden_sizes[layer_to_expand] += neurons_to_add
    plasticity_neurogenesis = BayesianFNN(784, expanded_hidden_sizes, 10).to(device)
    return plasticity_neurogenesis, expanded_hidden-sizes

In [150]:
def uncertainty_based_neurogenesis(plasticity_original, hidden_sizes, neurons_to_add=16):
    uncertainty = plasticity_original.get_average_uncertainty_per_layer()
    print("\n Average Uncertainty per Hidden Layer:")
    for i, val in enumerate(uncertainty):
        print(f"  Layer {i+1}: {val.item():.4f}")
    layer_to_expand = uncertainty.index(max(uncertainty[1:]))
    print(f"Expanding Layer {layer_to_expand+1} "
          f"(Highest Uncertainty: {uncertainty[layer_to_expand].item():.4f}) "
          f"by {neurons_to_add} neurons")
    expanded_hidden_sizes = hidden_sizes.copy()
    expanded_hidden_sizes[layer_to_expand] += neurons_to_add
    plasticity_neurogenesis = BayesianFNN(784, expanded_hidden_sizes, 10).to(device)
    return plasticity_neurogenesis, expanded_hidden_sizes

In [151]:
def expand_and_load_encoder_layer(old_sd, new_layer):
    new_sd = new_layer.state_dict()
    for k in new_sd.keys():
        if k not in old_sd:
            print(f"[skip] {k} not found in old layer")
            continue

        old_param = old_sd[k]
        new_param = new_sd[k]

        if old_param.shape == new_param.shape:
            new_sd[k] = old_param
        elif len(old_param.shape) == 2:
            # Linear weights: expand top-left corner
            new_sd[k][:old_param.shape[0], :old_param.shape[1]] = old_param
        elif len(old_param.shape) == 1:
            # Bias / LayerNorm
            new_sd[k][:old_param.shape[0]] = old_param
        else:
            print(f"[warn] Shape mismatch for {k}: old {old_param.shape}, new {new_param.shape}")

    new_layer.load_state_dict(new_sd, strict=False)

In [152]:
def run_experiment(experiment_name, model, train_loader, val_loader, test_loader, num_epochs, 
                   learning_rate=0.001, start_epoch=1, return_model=False, early_stopper=None, metrics=None, rewind=None):
    """Run a complete training experiment and return metrics"""
    print(f"\n{'-'*20} Running {experiment_name} experiment {'-'*20}")
    
    # Display model parameters
    param_stats = model.get_param_stats() if hasattr(model, 'get_param_stats') else {
        'total_params': sum(p.numel() for p in model.parameters()),
        'trainable_params': sum(p.numel() for p in model.parameters() if p.requires_grad)
    }
    
    print(f"Model parameters: {param_stats['total_params']:,}")
    print(f"Trainable parameters: {param_stats.get('trainable_params', param_stats['total_params']):,}")
    
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
    rewind_state = None
    
    # Track metrics
    if not metrics:
        metrics = {}
        metrics['train_loss_total'] = []
        metrics['train_loss_nll'] = []
        metrics['train_loss_kl'] = []
        metrics['train_acc'] = []
        metrics['val_loss_total'] = []
        metrics['val_loss_nll'] = []
        metrics['val_loss_kl'] = []
        metrics['val_acc'] = []

    best_acc = 0
    best_model_state = None
    # Training loop
    for epoch in range(start_epoch, start_epoch + num_epochs):
        # Train
        train_loss_total, train_acc, train_loss_nll, train_loss_kl = train(model, train_loader, optimizer, epoch, device)
        metrics['train_loss_total'].append(train_loss_total)
        metrics['train_loss_nll'].append(train_loss_nll)
        metrics['train_loss_kl'].append(train_loss_kl)
        metrics['train_acc'].append(train_acc)
        
        # Validate
        val_loss_total, val_acc, val_loss_nll, val_loss_kl= validate(model, val_loader, device)
        metrics['val_loss_total'].append(val_loss_total)
        metrics['val_loss_nll'].append(val_loss_nll)
        metrics['val_loss_kl'].append(val_loss_kl)
        metrics['val_acc'].append(val_acc)
        
        print(f'Epoch {epoch}: Train Loss={train_loss_total:.4f}, Train Acc={train_acc:.2f}%, '
              f'Val Loss={val_loss_total:.4f}, Val Acc={val_acc:.2f}%, ')
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save(best_model_state, f'./results/{experiment_name}/best_model.pth')

        if epoch - start_epoch + 1 == rewind:
            rewind_state = copy.deepcopy(model.state_dict())
        
        if early_stopper:
            early_stopper.check_early_stop(val_loss_total)
            if early_stopper.stop_training:
                num_epochs = epoch
                break

            

    # Plot and save metrics
    plot_metrics(
        {experiment_name: {
            'train_loss_total': metrics['train_loss_total'],
            'train_loss_nll': metrics['train_loss_nll'],
            'train_loss_kl': metrics['train_loss_kl'],
            'train_acc': metrics['train_acc'],
            'val_loss_total': metrics['val_loss_total'],
            'val_loss_nll': metrics['val_loss_nll'],
            'val_loss_kl': metrics['val_loss_kl'],
            'val_acc': metrics['val_acc']
        }}, 
        save_path=f'./results/{experiment_name}/metrics.png'
    )
    
    # Load best model for test
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print("Loaded best model based on validation accuracy for final testing.")

    test_loss_total, test_acc, test_loss_nll, test_loss_kl = validate(model, test_loader, device)

    print(f'Test Loss={test_loss_total:.4f}, Test Acc={test_acc:.2f}%')
    
    # Save model
    torch.save(model.state_dict(), f'./results/{experiment_name}/model.pth')
    
    # Update metrics
    metrics.update({
        'test_acc': test_acc,
        'test_loss_total': test_loss_total,
        'test_loss_nll': test_loss_nll,
        'test_loss_kl': test_loss_kl,
        'param_count': param_stats['total_params'],
        'trainable_param_count': param_stats.get('trainable_params', param_stats['total_params']),
    })
    
    # Create a metrics DataFrame
    metrics_df = pd.DataFrame({
        'epoch': range(1, 1 + len(metrics['train_loss_total'])),
        'train_loss_total': metrics['train_loss_total'],
        'train_loss_nll': metrics['train_loss_nll'],
        'train_loss_kl': metrics['train_loss_kl'],
        'train_acc': metrics['train_acc'],
        'val_loss_total': metrics['val_loss_total'],
        'val_loss_nll': metrics['val_loss_nll'],
        'val_loss_kl': metrics['val_loss_kl'],
        'val_acc': metrics['val_acc'],
    })
    metrics_df.to_csv(f'./results/{experiment_name}/metrics.csv', index=False)
    
    # Print summary
    print(f"\n{experiment_name} Summary:")
    print(f"Best validation accuracy: {max(metrics['val_acc'][start_epoch-1:]):.2f}%")
    print(f"Best validation loss: {min(metrics['val_loss_total'][start_epoch-1:]):.4f}")
    print(f"Final test accuracy: {test_acc:.2f}%")
    print(f"Parameter count: {param_stats['total_params']:,}")
    print(f"Trainable parameter count: {param_stats.get('trainable_params', param_stats['total_params']):,}")
    
    return metrics, model, num_epochs, rewind_state



In [185]:
def main():
    # Hyperparameters
    num_epochs = 50
    batch_size = 256
    learning_rate = 0.01
    hidden_sizes = [10,10,10,10]
    
    # Create results directory
    os.makedirs('results', exist_ok=True)
    

    # Create datasets
    transform = transforms.Compose([
        transforms.ToTensor(),  
        transforms.Lambda(lambda x: x.view(-1)) 
    ])
    
    training_data = datasets.FashionMNIST(
        root="../../Datasets",
        train=True,
        download=True,
        transform=transform
    )
    
    train_size = int(0.8 * len(training_data))
    val_size = len(training_data) - train_size 
    
    train_dataset, val_dataset = random_split(training_data, [train_size, val_size])

    test_dataset = datasets.FashionMNIST(
        root="../../Datasets",
        train=False,
        download=True,
        transform=transform
    )
    
    # Create data loaders with fixed seeds for workers
    g = torch.Generator()
    g.manual_seed(SEED)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=4,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=g
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=4,
        worker_init_fn=seed_worker,
        generator=g
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=4,
        worker_init_fn=seed_worker,
        generator=g
    )
    
    # ========== Experiment 1: Baseline Model ==========
    # print("\n\n" + "="*50)
    # print("EXPERIMENT 1: Training Baseline Model")
    # print("="*50)
    
    # baseline_model = BayesianFNN(784, hidden_sizes, 10).to(device)
    # baseline_metrics, baseline_model, _, _ = run_experiment(
    #     'baseline', 
    #     baseline_model, 
    #     train_loader, 
    #     val_loader, 
    #     test_loader, 
    #     num_epochs, 
    #     learning_rate,
    #     start_epoch=1,
    #     return_model=True
    #     #early_stopper=EarlyStopping()
    # )
    
    # ========== Experiment 2: Plasticity Model ==========
    print("\n\n" + "="*50)
    print("EXPERIMENT 2: Training Plasticity Model")
    print("="*50)
    print("+"*20 + " Growing Phase " + "+"*20)
    
    plasticity_model = BayesianFNN(784, hidden_sizes, 10).to(device)
    current_hidden_sizes = hidden_sizes
    plasticity_metrics = None
    prev_val_loss = float("inf")
    num_epochs_used = 0
    
    while(True):
        plasticity_metrics, plasticity_model, num_epochs_used, rewind_state = run_experiment(
            'plasticity', 
            plasticity_model, 
            train_loader, 
            val_loader, 
            test_loader, 
            num_epochs - num_epochs_used, 
            learning_rate,
            start_epoch=1 + num_epochs_used,
            return_model=True,
            early_stopper = EarlyStopping(),
            metrics = plasticity_metrics,
            rewind=1
        )
        current_best_val_loss = min(plasticity_metrics["val_loss_total"])
        if current_best_val_loss >= prev_val_loss:
            break
        prev_val_loss = current_best_val_loss
        old_model = plasticity_model
        new_model, current_hidden_sizes = uncertainty_based_neurogenesis(old_model, current_hidden_sizes, neurons_to_add=2)
        expand_and_load_encoder_layer(rewind_state, new_model)
        print(f"Average snr per layer: {plasticity_model.get_average_snr_per_layer()}")
        plasticity_model = new_model
        
    print("-"*20 + " Pruning Phase " + "-"*20)
        
    return plasticity_model
    
    
    # dropin_frozen_model = ResNet18(input_c=1, dropin=True, num_classes=2).to(device)
    
    # # Load baseline parameters to dropin model, handling shape mismatches properly
    # print("Loading parameters from baseline model to dropin_frozen model...")
    # params_loaded, params_skipped = load_parameters_from_baseline(baseline_model, dropin_frozen_model)
    
    # # Freeze original layers after loading parameters
    # dropin_frozen_model.freeze_original_layers()
    
    # # Train only the dropin layers
    # dropin_frozen_metrics = run_experiment(
    #     'dropin_frozen', 
    #     dropin_frozen_model, 
    #     train_loader, 
    #     val_loader, 
    #     test_loader, 
    #     num_epochs, 
    #     learning_rate,
    #     start_epoch=1
    # )
    
    # # ========== Experiment 3: Dropin Model with All Layers Trainable (from scratch) ==========
    # print("\n\n" + "="*50)
    # print("EXPERIMENT 3: Training Dropin Model with All Layers Trainable (from scratch)")
    # print("="*50)
    
    # dropin_unfrozen_model = ResNet18(input_c=1, dropin=True, num_classes=2).to(device)
    
    # # Make sure all layers are trainable
    # if hasattr(dropin_unfrozen_model, 'unfreeze_all_layers'):
    #     dropin_unfrozen_model.unfreeze_all_layers()
    
    # dropin_unfrozen_metrics = run_experiment(
    #     'dropin_unfrozen', 
    #     dropin_unfrozen_model, 
    #     train_loader, 
    #     val_loader, 
    #     test_loader, 
    #     num_epochs, 
    #     learning_rate,
    #     start_epoch=1
    # )
    
    # ========== Compare Results ==========
    
    # Create summary table
    # summary = pd.DataFrame([
    #     {
    #         'Model': 'Baseline',
    #         'Parameters': baseline_metrics['param_count'],
    #         'Trainable Params': baseline_metrics['trainable_param_count'],
    #         'Best Val Acc': max(baseline_metrics['val_acc']),
    #         'Test Acc': baseline_metrics['test_acc'],
    #     }
    # ])
    
    # summary.to_csv('./results/experiment_summary.csv', index=False)
    # print("\nExperiment Summary:")
    # print(summary)
    # return baseline_metrics

In [186]:
main()



EXPERIMENT 2: Training Plasticity Model
++++++++++++++++++++ Growing Phase ++++++++++++++++++++

-------------------- Running plasticity experiment --------------------
Model parameters: 16,580
Trainable parameters: 16,580


Validating: 100%|███████████████████████████████████████████████████████████████████████| 47/47 [00:01<00:00, 41.05it/s]


Epoch 1: Train Loss=1.7413, Train Acc=40.85%, Val Loss=2.1385, Val Acc=59.71%, 


Validating: 100%|███████████████████████████████████████████████████████████████████████| 47/47 [00:00<00:00, 62.38it/s]


Epoch 2: Train Loss=1.1031, Train Acc=66.76%, Val Loss=1.6672, Val Acc=72.42%, 


...

Validating: 100%|███████████████████████████████████████████████████████████████████████| 47/47 [00:01<00:00, 41.44it/s]


Epoch 29: Train Loss=0.6576, Train Acc=82.30%, Val Loss=1.2103, Val Acc=80.38%, 
Stopping early as no improvement has been observed.
Loaded best model based on validation accuracy for final testing.


Validating: 100%|███████████████████████████████████████████████████████████████████████| 40/40 [00:00<00:00, 46.49it/s]


Test Loss=1.3789, Test Acc=80.99%

plasticity Summary:
Best validation accuracy: 82.36%
Best validation loss: 1.1957
Final test accuracy: 80.99%
Parameter count: 16,664
Trainable parameter count: 16,664
-------------------- Pruning Phase --------------------


BayesianFNN(
  (layers): ModuleList(
    (0-3): 4 x BayesianLinear()
  )
  (out): BayesianLinear()
)

In [188]:
for i, layer in enumerate(m.layers):
    if i != 0:
        x = layer.get_snr()
        print(torch.mean(x,dim=1))

tensor([11.6302,  9.3140, 12.8132,  2.0072, 23.2114, 18.7555,  9.7945,  1.3802,
         6.9189,  7.8542,  1.0582,  6.8564], device='cuda:0',
       grad_fn=<MeanBackward1>)
tensor([ 9.9624, 16.9772, 13.2900, 11.0160,  6.2315,  8.2331,  1.5685,  9.0278,
        11.7514,  4.8277], device='cuda:0', grad_fn=<MeanBackward1>)
tensor([11.0856,  4.7006,  2.1034, 15.0946, 15.3131, 12.3587,  9.9595, 14.0852,
        11.0122, 17.0094], device='cuda:0', grad_fn=<MeanBackward1>)


below 5 snr?
identify the below 5 snr and create a mask (maybe dictionary?) {1: [2,3,7], 2 []1,2,3} first hidden layer the neuron 2,3,7 below 5 snr
then with that kind know the architecture of the pruned model, initialise that, then create a function to transfer the weights over (rewind to checkpoint?)